# 2. 多租户 Semantic Cache：怎样既复用相似问题，又不泄露其他租户答案？

## 面试回答主线

Semantic cache 不能只用 query embedding 做全局近邻搜索，因为相似问题可能属于不同租户、权限范围、模型版本或政策版本。安全 cache key 应先按 tenant、auth scope、model、policy 与有效期做硬过滤，再在同一隔离分区内计算语义相似度。面试时我会用真实退款、发票和账号问题手写 TF-IDF embedding 与 cosine，先复现全局缓存把甲公司答案泄露给乙公司，再展示隔离查询。相似度命中还要有语义护栏，例如否定词、时间敏感意图与工具副作用请求不能直接复用。缓存条目应保留来源、生成配置和失效条件，便于审计与主动 invalidation。生产系统还需考虑 PII 加密、容量淘汰、embedding 漂移与租户级命中率。

## 1. 真实案例：两个租户的退款、发票和账号政策

Alpha 与 Beta 的问题措辞相似，但退款期限、发票流程和安全策略不同。六个缓存条目与六个新请求都带 tenant、scope、model、policy 和 expires_at；token 列表来自可读中文关键词。

In [1]:
from pprint import pprint  # 导入结构化打印工具展示多租户缓存数据
import math  # 导入对数与平方根函数手写 TF-IDF 和余弦相似度
entries = [{"cache_id": "C01", "tenant": "alpha", "scope": "support", "model": "m1", "policy": "A-2026", "tokens": ["订单", "退款", "期限"], "answer": "Alpha 未发货订单 7 天内可退", "expires_at": 200}, {"cache_id": "C02", "tenant": "beta", "scope": "support", "model": "m1", "policy": "B-2026", "tokens": ["订单", "退款", "期限"], "answer": "Beta 企业订单 30 天内可退", "expires_at": 200}, {"cache_id": "C03", "tenant": "alpha", "scope": "finance", "model": "m1", "policy": "A-2026", "tokens": ["发票", "抬头", "修改"], "answer": "Alpha 在开票前可在线修改抬头", "expires_at": 200}, {"cache_id": "C04", "tenant": "beta", "scope": "finance", "model": "m1", "policy": "B-2026", "tokens": ["发票", "抬头", "修改"], "answer": "Beta 需提交财务工单修改抬头", "expires_at": 200}, {"cache_id": "C05", "tenant": "alpha", "scope": "account", "model": "m1", "policy": "A-2026", "tokens": ["账号", "密码", "重置"], "answer": "Alpha 可通过企业邮箱重置密码", "expires_at": 200}, {"cache_id": "C06", "tenant": "beta", "scope": "account", "model": "m1", "policy": "B-2026", "tokens": ["账号", "密码", "重置"], "answer": "Beta 必须由管理员发起密码重置", "expires_at": 200}]  # 定义六条租户政策不同的真实缓存
requests = [{"id": "S01", "tenant": "alpha", "scope": "support", "model": "m1", "policy": "A-2026", "tokens": ["订单", "退款", "期限"], "expected": "C01"}, {"id": "S02", "tenant": "beta", "scope": "support", "model": "m1", "policy": "B-2026", "tokens": ["退款", "订单", "期限"], "expected": "C02"}, {"id": "S03", "tenant": "alpha", "scope": "finance", "model": "m1", "policy": "A-2026", "tokens": ["修改", "发票", "抬头"], "expected": "C03"}, {"id": "S04", "tenant": "beta", "scope": "finance", "model": "m1", "policy": "B-2026", "tokens": ["发票", "修改", "抬头"], "expected": "C04"}, {"id": "S05", "tenant": "alpha", "scope": "account", "model": "m1", "policy": "A-2026", "tokens": ["重置", "账号", "密码"], "expected": "C05"}, {"id": "S06", "tenant": "beta", "scope": "account", "model": "m1", "policy": "B-2026", "tokens": ["账号", "重置", "密码"], "expected": "C06"}]  # 定义六个需要安全复用的真实请求
now = 100  # 设置确定性的当前时间用于 TTL 检查
preview = [{"请求": item["id"], "tenant": item["tenant"], "scope": item["scope"], "关键词": item["tokens"], "期望缓存": item["expected"]} for item in requests]  # 汇总多租户查询字段
print("多租户 Semantic Cache 输入预览：")  # 输出真实案例标题
pprint(preview, sort_dicts=False)  # 展示相似问题背后的租户与权限差异

多租户 Semantic Cache 输入预览：
[{'请求': 'S01',
  'tenant': 'alpha',
  'scope': 'support',
  '关键词': ['订单', '退款', '期限'],
  '期望缓存': 'C01'},
 {'请求': 'S02',
  'tenant': 'beta',
  'scope': 'support',
  '关键词': ['退款', '订单', '期限'],
  '期望缓存': 'C02'},
 {'请求': 'S03',
  'tenant': 'alpha',
  'scope': 'finance',
  '关键词': ['修改', '发票', '抬头'],
  '期望缓存': 'C03'},
 {'请求': 'S04',
  'tenant': 'beta',
  'scope': 'finance',
  '关键词': ['发票', '修改', '抬头'],
  '期望缓存': 'C04'},
 {'请求': 'S05',
  'tenant': 'alpha',
  'scope': 'account',
  '关键词': ['重置', '账号', '密码'],
  '期望缓存': 'C05'},
 {'请求': 'S06',
  'tenant': 'beta',
  'scope': 'account',
  '关键词': ['账号', '重置', '密码'],
  '期望缓存': 'C06'}]


## 2. 手写 TF-IDF embedding 与全局近邻 Baseline（基线）

先从缓存与请求共同语料计算 IDF，再把 token 计数乘 IDF 得到稀疏向量。错误基线在所有租户条目中直接选择最大 cosine；相同措辞会发生并列，并被列表中先出现的 Alpha 条目截获。

In [2]:
all_documents = [item["tokens"] for item in entries] + [item["tokens"] for item in requests]  # 收集缓存和在线请求共同估计词频
vocabulary = sorted(set(token for document in all_documents for token in document))  # 构造可复现的中文关键词词表
document_frequency = {token: sum(token in document for document in all_documents) for token in vocabulary}  # 统计每个关键词出现于多少条文本
idf = {token: math.log((1 + len(all_documents)) / (1 + document_frequency[token])) + 1.0 for token in vocabulary}  # 手写平滑 IDF 避免零除
def embed(tokens):  # 将一组中文关键词转换为 TF-IDF 稀疏向量
    return [tokens.count(token) * idf[token] for token in vocabulary]  # 按固定词表顺序计算词频乘逆文档频率
def cosine(left, right):  # 手写两个 TF-IDF 向量的余弦相似度
    numerator = sum(a * b for a, b in zip(left, right))  # 计算向量内积
    left_norm = math.sqrt(sum(value * value for value in left))  # 计算左向量 L2 范数
    right_norm = math.sqrt(sum(value * value for value in right))  # 计算右向量 L2 范数
    return numerator / (left_norm * right_norm) if left_norm and right_norm else 0.0  # 返回归一化相似度并处理空向量
entry_vectors = {entry["cache_id"]: embed(entry["tokens"]) for entry in entries}  # 预计算六条缓存 embedding
def global_lookup(request):  # 实现忽略租户隔离的错误全局语义搜索
    request_vector = embed(request["tokens"])  # 计算当前在线问题的 TF-IDF 表示
    scored = [(entry, cosine(request_vector, entry_vectors[entry["cache_id"]])) for entry in entries if entry["expires_at"] > now]  # 对全部未过期租户缓存计算相似度
    return max(scored, key=lambda pair: pair[1])  # 直接返回全局最相似条目导致跨租户风险
baseline_rows = []  # 收集六个请求的全局缓存结果
for request in requests:  # 遍历同一批多租户问题
    entry, score = global_lookup(request)  # 执行没有隔离过滤的语义近邻基线
    leak = entry["tenant"] != request["tenant"] or entry["scope"] != request["scope"]  # 检查答案是否来自错误安全边界
    baseline_rows.append({"请求": request["id"], "请求tenant": request["tenant"], "命中": entry["cache_id"], "命中tenant": entry["tenant"], "答案": entry["answer"], "相似度": round(score, 3), "跨边界": leak})  # 保存逐请求泄露证据
print("错误的全局 Semantic Cache 基线：")  # 标注当前输出属于无隔离基线
pprint(baseline_rows, sort_dicts=False)  # 展示相同措辞如何命中其他租户政策

错误的全局 Semantic Cache 基线：
[{'请求': 'S01',
  '请求tenant': 'alpha',
  '命中': 'C01',
  '命中tenant': 'alpha',
  '答案': 'Alpha 未发货订单 7 天内可退',
  '相似度': 1.0,
  '跨边界': False},
 {'请求': 'S02',
  '请求tenant': 'beta',
  '命中': 'C01',
  '命中tenant': 'alpha',
  '答案': 'Alpha 未发货订单 7 天内可退',
  '相似度': 1.0,
  '跨边界': True},
 {'请求': 'S03',
  '请求tenant': 'alpha',
  '命中': 'C03',
  '命中tenant': 'alpha',
  '答案': 'Alpha 在开票前可在线修改抬头',
  '相似度': 1.0,
  '跨边界': False},
 {'请求': 'S04',
  '请求tenant': 'beta',
  '命中': 'C03',
  '命中tenant': 'alpha',
  '答案': 'Alpha 在开票前可在线修改抬头',
  '相似度': 1.0,
  '跨边界': True},
 {'请求': 'S05',
  '请求tenant': 'alpha',
  '命中': 'C05',
  '命中tenant': 'alpha',
  '答案': 'Alpha 可通过企业邮箱重置密码',
  '相似度': 1.0,
  '跨边界': False},
 {'请求': 'S06',
  '请求tenant': 'beta',
  '命中': 'C05',
  '命中tenant': 'alpha',
  '答案': 'Alpha 可通过企业邮箱重置密码',
  '相似度': 1.0,
  '跨边界': True}]


## 3. 核心机制：先做安全合同硬过滤，再计算语义相似度

候选必须同时满足 tenant、scope、model、policy 与 TTL，任一字段不同就不能进入向量排序。语义阈值只在隔离分区内部生效；没有合格候选时应 miss 并重新生成，而不是扩大到全局。

In [3]:
def safe_lookup(request, threshold=0.75):  # 在硬隔离合同内执行语义缓存查询
    candidates = [entry for entry in entries if entry["tenant"] == request["tenant"] and entry["scope"] == request["scope"] and entry["model"] == request["model"] and entry["policy"] == request["policy"] and entry["expires_at"] > now]  # 按全部安全维度过滤候选
    request_vector = embed(request["tokens"])  # 计算当前问题的 TF-IDF 向量
    scored = [(entry, cosine(request_vector, entry_vectors[entry["cache_id"]])) for entry in candidates]  # 只在隔离分区内计算语义相似度
    if not scored:  # 当前租户合同下没有任何有效缓存
        return None, 0.0, candidates  # 返回 cache miss 而不尝试跨边界回退
    best_entry, best_score = max(scored, key=lambda pair: pair[1])  # 选择安全候选中的最高相似度条目
    if best_score < threshold:  # 检查相似度是否达到复用质量门槛
        return None, best_score, candidates  # 低相似问题必须重新调用模型生成
    return best_entry, best_score, candidates  # 返回通过硬合同和软相似度的缓存命中
sample_request = requests[1]  # 选择容易被全局基线泄露的 Beta 退款请求
sample_entry, sample_score, sample_candidates = safe_lookup(sample_request)  # 执行安全过滤与相似度排序
print({"请求": sample_request["id"], "硬过滤后候选": [entry["cache_id"] for entry in sample_candidates], "最终命中": sample_entry["cache_id"], "相似度": round(sample_score, 3), "答案": sample_entry["answer"]})  # 展示先隔离再相似搜索的关键中间量

{'请求': 'S02', '硬过滤后候选': ['C02'], '最终命中': 'C02', '相似度': 1.0, '答案': 'Beta 企业订单 30 天内可退'}


## 4. 逐请求安全缓存结果与命中证据

下面在六个请求上保存候选数量、命中 ID、来源租户和是否符合期望。每个分区恰好一个当前政策条目，但代码仍执行真实 embedding 与阈值判断。

In [4]:
safe_rows = []  # 收集多租户安全查询的逐请求结果
for request in requests:  # 遍历六个真实业务问题
    entry, score, candidates = safe_lookup(request)  # 先过滤安全合同再做语义近邻
    hit_id = entry["cache_id"] if entry else None  # 提取命中缓存 ID 或明确 miss
    safe_rows.append({"请求": request["id"], "tenant": request["tenant"], "隔离候选数": len(candidates), "命中": hit_id, "答案": entry["answer"] if entry else "重新生成", "相似度": round(score, 3), "符合期望": hit_id == request["expected"]})  # 保存逐请求命中与来源证据
print("隔离 Semantic Cache 逐请求结果：")  # 输出核心方案结果标题
pprint(safe_rows, sort_dicts=False)  # 展示两个租户分别获得自己的政策答案

隔离 Semantic Cache 逐请求结果：
[{'请求': 'S01',
  'tenant': 'alpha',
  '隔离候选数': 1,
  '命中': 'C01',
  '答案': 'Alpha 未发货订单 7 天内可退',
  '相似度': 1.0,
  '符合期望': True},
 {'请求': 'S02',
  'tenant': 'beta',
  '隔离候选数': 1,
  '命中': 'C02',
  '答案': 'Beta 企业订单 30 天内可退',
  '相似度': 1.0,
  '符合期望': True},
 {'请求': 'S03',
  'tenant': 'alpha',
  '隔离候选数': 1,
  '命中': 'C03',
  '答案': 'Alpha 在开票前可在线修改抬头',
  '相似度': 1.0,
  '符合期望': True},
 {'请求': 'S04',
  'tenant': 'beta',
  '隔离候选数': 1,
  '命中': 'C04',
  '答案': 'Beta 需提交财务工单修改抬头',
  '相似度': 1.0,
  '符合期望': True},
 {'请求': 'S05',
  'tenant': 'alpha',
  '隔离候选数': 1,
  '命中': 'C05',
  '答案': 'Alpha 可通过企业邮箱重置密码',
  '相似度': 1.0,
  '符合期望': True},
 {'请求': 'S06',
  'tenant': 'beta',
  '隔离候选数': 1,
  '命中': 'C06',
  '答案': 'Beta 必须由管理员发起密码重置',
  '相似度': 1.0,
  '符合期望': True}]


## 5. 结果解读：命中率之外必须报告跨租户错误率

全局基线表面上六条都有 cache hit，但其中 Beta 请求会拿到 Alpha 政策，命中率指标掩盖了安全事故。安全方案仍命中六条，同时跨边界为零；生产仪表盘要分别统计 safe hit、miss 和 isolation violation。

In [5]:
global_leaks = sum(row["跨边界"] for row in baseline_rows)  # 统计全局缓存返回错误租户或权限答案的次数
safe_correct = sum(row["符合期望"] for row in safe_rows)  # 统计硬隔离后命中正确政策缓存的次数
safe_cross_boundary = sum(row["tenant"] != next(entry["tenant"] for entry in entries if entry["cache_id"] == row["命中"]) for row in safe_rows if row["命中"])  # 复核安全方案是否存在租户来源不一致
comparison = [{"请求": request["id"], "全局命中": baseline_rows[index]["命中"], "全局跨边界": baseline_rows[index]["跨边界"], "安全命中": safe_rows[index]["命中"], "安全正确": safe_rows[index]["符合期望"]} for index, request in enumerate(requests)]  # 构造同数据逐请求对照
print("多租户缓存逐请求对照：")  # 输出结果解读标题
pprint(comparison, sort_dicts=False)  # 展示全局 hit 与安全 hit 的本质差异
print({"全局缓存跨边界次数": global_leaks, "安全缓存正确命中": f"{safe_correct}/{len(requests)}", "安全方案跨边界次数": safe_cross_boundary})  # 汇总安全优先指标

多租户缓存逐请求对照：
[{'请求': 'S01', '全局命中': 'C01', '全局跨边界': False, '安全命中': 'C01', '安全正确': True},
 {'请求': 'S02', '全局命中': 'C01', '全局跨边界': True, '安全命中': 'C02', '安全正确': True},
 {'请求': 'S03', '全局命中': 'C03', '全局跨边界': False, '安全命中': 'C03', '安全正确': True},
 {'请求': 'S04', '全局命中': 'C03', '全局跨边界': True, '安全命中': 'C04', '安全正确': True},
 {'请求': 'S05', '全局命中': 'C05', '全局跨边界': False, '安全命中': 'C05', '安全正确': True},
 {'请求': 'S06', '全局命中': 'C05', '全局跨边界': True, '安全命中': 'C06', '安全正确': True}]
{'全局缓存跨边界次数': 3, '安全缓存正确命中': '6/6', '安全方案跨边界次数': 0}


## 6. 失败案例与修正：否定问题与肯定问题 embedding 仍很相似

“不要为订单退款”与“订单退款期限”共享两个高权重关键词，单靠 cosine 可能错误复用退款指导。修正是在 embedding 命中后做语义 guard：否定极性、工具副作用和时间敏感字段不一致时强制 miss。

In [6]:
negative_request = {"id": "NEG", "tenant": "alpha", "scope": "support", "model": "m1", "policy": "A-2026", "tokens": ["不要", "订单", "退款"]}  # 构造与退款缓存高相似但意图相反的请求
unsafe_entry, unsafe_score, unsafe_candidates = safe_lookup(negative_request, threshold=0.40)  # 复现只靠较宽语义阈值产生的错误命中
def polarity_guard(request_tokens, entry_tokens):  # 定义缓存命中后的最小否定极性检查
    request_negative = any(token in {"不要", "禁止", "取消"} for token in request_tokens)  # 判断在线请求是否包含明确否定或取消意图
    entry_negative = any(token in {"不要", "禁止", "取消"} for token in entry_tokens)  # 判断缓存原问题是否具有相同极性
    return request_negative == entry_negative  # 只有极性一致才允许复用已有答案
guard_passed = polarity_guard(negative_request["tokens"], unsafe_entry["tokens"]) if unsafe_entry else False  # 对高相似候选执行否定语义门禁
fixed_decision = unsafe_entry["cache_id"] if unsafe_entry and guard_passed else "MISS_AND_REGENERATE"  # 极性冲突时拒绝缓存并重新生成
print({"失败_仅相似度命中": unsafe_entry["cache_id"] if unsafe_entry else None, "相似度": round(unsafe_score, 3), "缓存答案": unsafe_entry["answer"] if unsafe_entry else None, "极性检查通过": guard_passed, "修正决策": fixed_decision})  # 展示 false hit 与语义 guard 修正

{'失败_仅相似度命中': 'C01', '相似度': 0.816, '缓存答案': 'Alpha 未发货订单 7 天内可退', '极性检查通过': False, '修正决策': 'MISS_AND_REGENERATE'}


## 7. 生产差距与最小回归检查

生产缓存还应把 temperature、tool schema、locale、数据权限快照和生成模型指纹纳入合同，并在政策更新时主动失效旧条目。向量索引、答案正文和日志都要租户级加密与访问控制；阈值需按意图校准，不能全局固定。下面的断言只验证真实样本中的隔离、TTL、命中结果和否定失败修正。

In [7]:
assert len(requests) >= 6  # 确认真实多租户请求数量满足逐样本教学要求
assert global_leaks > 0  # 确认只做全局向量近邻真实产生跨租户风险
assert safe_correct == len(requests)  # 确认硬合同过滤后六个请求命中各自政策缓存
assert safe_cross_boundary == 0  # 确认安全方案没有返回其他租户条目
assert all(entry["expires_at"] > now for entry in entries)  # 确认主实验命中来自仍在 TTL 内的缓存
assert unsafe_entry is not None and guard_passed is False  # 确认否定请求可骗过相似度但被极性门禁拒绝
assert fixed_decision == "MISS_AND_REGENERATE"  # 确认语义冲突不会静默复用旧答案
print("回归检查通过：租户隔离、语义命中、TTL 与否定意图 guard 均已验证。")  # 输出最终验收结论

回归检查通过：租户隔离、语义命中、TTL 与否定意图 guard 均已验证。
